In [1]:
import pandas as pd
import numpy as np
from math import e
import matplotlib.pyplot as plt

pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_rows", 200)
pd.set_option("display.max_columns", None)


column_types = {
    "id": "Int64",
    "osm_id": "str",
    "addr_street": "str",
    "addr_number": "str",
    "ground_floor_group": "str",
    "ground_floor_function": "str",
    "upper_floor_group": "str",
    "upper_floor_function": "str",
    "number_of_storeys": "Int64",
    "is_mixed_use": "boolean",
    "build_year_est": "Int64",
    "arch_style": "str",
    "condition": "Int64",
    "max_height": "float64",
    "is_heritage": "str",
    "source": "str",
    "confidence": "Int64",
    "date_of_checking": "str",
    "number_of_functions": "Int64",
}

In [2]:
data_gdynia = pd.read_csv(
    "../data/processed/fixed/srodmiescie.csv",
    sep=";",
    index_col=0,
    dtype=column_types,
    parse_dates=["date_of_checking"],
)

1.

In [3]:
cumulative_dates = (
    data_gdynia.resample("D", on="date_of_checking")
    .agg(
        num_buildings=("osm_id", "size"),
    )
    .assign(cum_buildings=lambda d: d["num_buildings"].cumsum())
    .assign(cum_share=lambda d: d["cum_buildings"].div(len(data_gdynia)))
)
cumulative_dates

,num_buildings,cum_buildings,cum_share
date_of_checking,,,
2026-09-23,52,52,0.052953
2026-09-24,108,160,0.162933
2026-09-25,67,227,0.231161
2026-09-26,100,327,0.332994
2026-09-27,461,788,0.802444
2026-09-28,0,788,0.802444
2026-09-29,22,810,0.824847
2026-09-30,172,982,1.000000


In [4]:
(cumulative_dates["cum_share"] > 0.50).idxmax()

Timestamp('2026-09-27 00:00:00')

2.

In [5]:
date_resample = (
    data_gdynia.resample("D", on="date_of_checking")
    .agg(
        num_buildings=("osm_id", "size"),
    )
    .assign(num_diff=lambda d: d["num_buildings"].diff())
)
date_resample

,num_buildings,num_diff
date_of_checking,,
2026-09-23,52,NaN
2026-09-24,108,56.0
2026-09-25,67,-41.0
2026-09-26,100,33.0
2026-09-27,461,361.0
2026-09-28,0,-461.0
2026-09-29,22,22.0
2026-09-30,172,150.0


In [6]:
data_gdynia.groupby("date_of_checking").agg(
    num_buildings=("osm_id", "size"),
).assign(num_diff=lambda d: d["num_buildings"].diff())

,num_buildings,num_diff
date_of_checking,,
2026-09-23,52,NaN
2026-09-24,108,56.0
2026-09-25,67,-41.0
2026-09-26,100,33.0
2026-09-27,461,361.0
2026-09-29,22,-439.0
2026-09-30,172,150.0


In [7]:
print(
    f"the biggest decline was: {date_resample.loc[date_resample['num_diff'].idxmin(), 'num_diff']} on the day of: {date_resample['num_diff'].idxmin()}"
)
print(
    f"the biggest increase was: {date_resample.loc[date_resample['num_diff'].idxmax(), 'num_diff']} on the day of: {date_resample['num_diff'].idxmax()}"
)

the biggest decline was: -461.0 on the day of: 2026-09-28 00:00:00
the biggest increase was: 361.0 on the day of: 2026-09-27 00:00:00


In [8]:
assert (
    date_resample["num_diff"].fillna(date_resample["num_buildings"].iloc[0]).cumsum()
    == date_resample["num_buildings"]
).all()

3.

In [9]:
date_resample_shift = (
    data_gdynia.resample("D", on="date_of_checking")
    .agg(
        num_buildings=("osm_id", "size"),
    )
    .assign(num_prev_day=lambda d: d["num_buildings"].shift())
    .assign(num_diff=lambda d: d["num_buildings"] - d["num_prev_day"])
    .assign(day_perc_change=lambda d: d["num_buildings"] / d["num_prev_day"] - 1)
    .assign(pct_change=lambda d: d["num_buildings"].pct_change())
)
date_resample_shift

,num_buildings,num_prev_day,num_diff,day_perc_change,pct_change
date_of_checking,,,,,
2026-09-23,52,NaN,NaN,NaN,NaN
2026-09-24,108,52.0,56.0,1.076923,1.076923
2026-09-25,67,108.0,-41.0,-0.379630,-0.379630
2026-09-26,100,67.0,33.0,0.492537,0.492537
2026-09-27,461,100.0,361.0,3.610000,3.610000
2026-09-28,0,461.0,-461.0,-1.000000,-1.000000
2026-09-29,22,0.0,22.0,inf,inf
2026-09-30,172,22.0,150.0,6.818182,6.818182


In [10]:
assert np.isinf(date_resample_shift["pct_change"]).sum() == 1
assert date_resample_shift["pct_change"].isna().sum() == 1

pd.testing.assert_series_equal(
    date_resample_shift["day_perc_change"],
    date_resample_shift["pct_change"],
    check_names=False,
)

pd.testing.assert_series_equal(
    date_resample_shift["num_diff"], date_resample["num_diff"]
)

4.

In [11]:
str_date_cumsum = (
    data_gdynia.groupby(["addr_street", "date_of_checking"])
    .size()
    .to_frame("num_buildings")
    .assign(num_cumsum=lambda d: d.groupby("addr_street").cumsum())
)
str_date_cumsum.tail()

num_buildings  num_cumsum
addr_street                        date_of_checking                           
Świętojańska                       2026-09-26                   10          11
                                   2026-09-27                   35          46
                                   2026-09-29                   11          57
                                   2026-09-30                   49         106
Żołnierzy I Armii Wojska Polskiego 2026-09-30                    3           3

In [12]:
assert (str_date_cumsum.loc["Świętojańska", "num_cumsum"] == [1, 11, 46, 57, 106]).all()
assert (
    data_gdynia.groupby(["addr_street", "date_of_checking"])
    .size()
    .to_frame("num_buildings")
    .cumsum()
    .loc["Świętojańska", "num_buildings"][-1:]
    == 710
).all()

assert (
    data_gdynia.groupby("addr_street")["date_of_checking"].nunique().eq(1).sum() == 16
)

pd.testing.assert_series_equal(
    data_gdynia.groupby(["addr_street", "date_of_checking"])
    .size()
    .to_frame("num_buildings")
    .groupby("addr_street")
    .cumsum()
    .groupby("addr_street")
    .tail(1)
    .sort_values("num_buildings", ascending=False)
    .reset_index()
    .set_index("addr_street")
    .loc[:, "num_buildings"]
    .sort_index(),
    data_gdynia["addr_street"].value_counts().sort_index(),
    check_names=False,
)


5.

In [ ]:
date_visit_diff = (
    data_gdynia.groupby(["addr_street", "date_of_checking"])
    .size()
    .reset_index(name="num_build")
    .assign(
        visit_diff=lambda d: d.groupby("addr_street")["date_of_checking"].diff().dt.days
    )
)
date_visit_diff.head()

,addr_street,date_of_checking,num_build,visit_diff
0,10 Lutego,2026-09-24,24,NaN
1,10 Lutego,2026-09-26,8,2.0
2,10 Lutego,2026-09-27,5,1.0
3,10 Lutego,2026-09-30,1,3.0
4,3 Maja,2026-09-23,6,NaN


In [ ]:
assert len(date_visit_diff) == 96
assert date_visit_diff.isna().sum().sum() == 44
assert (date_visit_diff.groupby("addr_street").size() >= 2).sum() == 28
assert date_visit_diff["visit_diff"].value_counts().to_dict() == {
    1: 24,
    2: 15,
    3: 11,
    4: 2,
}
assert (
    date_visit_diff.loc[date_visit_diff["visit_diff"] == 4, "addr_street"]
    == ["Antoniego Hryniewickiego", "Plac Konstytucji"]
).all()
assert (
    data_gdynia.groupby(["addr_street", "date_of_checking"])
    .size()
    .reset_index()
    .loc[:, "date_of_checking"]
    .diff()
    .dt.days.lt(0)
    .sum()
    == 27
)

6.

In [ ]:
date_visit_diff2 = date_visit_diff.assign(
    prev_visit=lambda d: d.groupby("addr_street")["date_of_checking"].shift()
).assign(visit_diff2=lambda d: (d["date_of_checking"] - d["prev_visit"]).dt.days)

In [ ]:
pd.testing.assert_series_equal(
    date_visit_diff2["visit_diff"], date_visit_diff2["visit_diff2"], check_names=False
)

assert (
    date_visit_diff2.loc[
        date_visit_diff2["addr_street"] == "Świętojańska", "date_of_checking"
    ]
    == pd.to_datetime(
        ["2026-09-24", "2026-09-26", "2026-09-27", "2026-09-29", "2026-09-30"]
    )
).all()


In [17]:
date_visit_diff2

,addr_street,date_of_checking,num_build,visit_diff,prev_visit,visit_diff2
0,10 Lutego,2026-09-24,24,NaN,NaT,NaN
1,10 Lutego,2026-09-26,8,2.0,2026-09-24,2.0
2,10 Lutego,2026-09-27,5,1.0,2026-09-26,1.0
3,10 Lutego,2026-09-30,1,3.0,2026-09-27,3.0
4,3 Maja,2026-09-23,6,NaN,NaT,NaN
5,3 Maja,2026-09-24,10,1.0,2026-09-23,1.0
6,3 Maja,2026-09-27,5,3.0,2026-09-24,3.0
7,Aleja Jana Pawła II,2026-09-24,1,NaN,NaT,NaN
8,Aleja Jana Pawła II,2026-09-27,14,3.0,2026-09-24,3.0
9,Aleja Marszałka Józefa Piłsudskiego,2026-09-30,2,NaN,NaT,NaN


In [ ]:
data_gdynia.groupby(['addr_street','arch'])